<a href="https://colab.research.google.com/github/shubhamparmarp70/ResNet-5-/blob/main/ResNet_5_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchvision numpy matplotlib scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set Dataset Path
dataset_path = "/content/drive/MyDrive/dataset"

In [ ]:
# Verify Dataset
import os

print(os.listdir(dataset_path))
print(os.listdir(dataset_path + "/train"))

['test', 'train']
['glasses', 'sunglasses-imagenet', 'unAugmented-images', 'sunglasses', 'plain', 'LNS Occluded', 'covering', 'Augmented-images']


In [ ]:
# Load Dataset in PyTorch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.ImageFolder(dataset_path + "/train", transform=transform)
test_data = datasets.ImageFolder(dataset_path + "/test", transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print("Classes:", train_data.classes)

Classes: ['Augmented-images', 'LNS Occluded', 'covering', 'glasses', 'plain', 'sunglasses', 'sunglasses-imagenet', 'unAugmented-images']


In [ ]:
# Add LeNet-5 Model
import torch.nn as nn
import torch.nn.functional as F

class LeNet5(nn.Module):
    def __init__(self, num_classes=5):
        super(LeNet5, self).__init__()

        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)

        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.avg_pool2d(x, 2)

        x = F.relu(self.conv2(x))
        x = F.avg_pool2d(x, 2)

        x = x.view(-1, 16*5*5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [ ]:
# Training Setup
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LeNet5(num_classes=len(train_data.classes)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Train Modele
pochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 252.1151
Epoch 2, Loss: 198.2568
Epoch 3, Loss: 177.1267
Epoch 4, Loss: 157.4736
Epoch 5, Loss: 147.3610
Epoch 6, Loss: 134.6404
Epoch 7, Loss: 126.3656
Epoch 8, Loss: 121.0001
Epoch 9, Loss: 115.4060
Epoch 10, Loss: 105.9714


In [ ]:
# Calculate Accuracy
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"✅ Test Accuracy: {accuracy:.2f}%")

✅ Test Accuracy: 88.87%


In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()

    correct = 0
    total = 0
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # 🔹 Calculate accuracy
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}")
    print(f"Loss: {running_loss:.4f}")
    print(f"✅ Training Accuracy: {train_accuracy:.2f}%")